# Event Weights

Calculate concurrency, average uniqueness, return attribution, time decay, and normalized sample weights from the labeled AAPL events. Development and holdout are processed independently while preserving the established 64-column weighted-event schema.


## Process the Data


In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.data_preprocessing.event_weights import (
    WEIGHT_COLUMNS,
    build_partitioned_event_weights,
)

period = "2025-01-01_2025-12-31"
event_dir = PROJECT_ROOT / "data/research_data/events"
event_path = event_dir / f"aapl_news_primary_model_{period}.parquet"
partition_path = event_dir / f"aapl_news_labeled_split_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"
weighted_path = event_dir / f"aapl_news_modeling_weighted_{period}.parquet"

events = pd.read_parquet(event_path).sort_values("event_start", ignore_index=True)
partition_manifest = pd.read_parquet(partition_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")
events["event_start"] = pd.to_datetime(events["event_start"], utc=True)
events["event_end"] = pd.to_datetime(events["event_end"], utc=True)
partition_manifest["event_start"] = pd.to_datetime(partition_manifest["event_start"], utc=True)
close = dollar_bars.set_index("end")["close"].astype(float)


In [ ]:
weighted_events = build_partitioned_event_weights(
    events,
    partition_manifest,
    close,
)
weight_columns = WEIGHT_COLUMNS

weighted_events.to_parquet(weighted_path, index=False)
print(weighted_path)


## Take a Quick Look at the Data Structure


In [ ]:
development_starts = partition_manifest.loc[partition_manifest["partition"].eq("development"), "event_start"]
development_data = weighted_events[weighted_events["event_start"].isin(development_starts)]
development_data.head()

In [ ]:
development_data.info()

In [ ]:
development_data["direction_label"].value_counts()

In [ ]:
development_data[weight_columns].describe()

In [ ]:
development_data[weight_columns].hist(figsize=(12, 8), bins=30)